# Missionários em segurança com canibais
O intuito desse trabalho é gerenciar o uso do barco para que todos os missionários da paróquia do Frei Sardinha consiga atravessar o rio em segurança na companhia dos queridos indígenas Caetés. Mas para garantir a segurança de todos, principalmente dos missionários, existem algumas regras que devem ser respeitadas.

Primeiramente, o problema considera duas margens do rio: a margem inicial (esquerda) e a margem de destino (direita). Todos os indivíduos começam na margem inicial e devem ser transportados para a margem de destino.

O transporte é realizado por meio de um barco, que possui capacidade limitada, podendo transportar no máximo um número fixo de pessoas por travessia.

## Regras

As regras que regem o problema são as seguintes:

- O barco deve sempre transportar pelo menos uma pessoa, o barco nunca se desloca sozinho.
- Em nenhuma das margens do rio pode haver um número de indígenas maior que o de missionários, caso exista pelo menos um missionário presente naquela margem. Caso contrário, os missionários seriam colocados em risco.

## Definições iniciais

In [141]:
TOTAL_MISSIONARIES = 4
TOTAL_CANNIBALS = 4
BOAT_CAPACITY = 3

initial_state = {
    "left": (TOTAL_MISSIONARIES, TOTAL_CANNIBALS),
    "right": (0, 0),
    "boat": "left"
}
initial_state

{'left': (4, 4), 'right': (0, 0), 'boat': 'left'}

### Validação
A validação consiste em verificar em cada margem, se há mais canibais que missionarios.

In [142]:
def is_valid(state):
    left_missionaries, left_cannibals = state["left"]
    right_missionaries, right_cannibals = state["right"]

    if min(left_missionaries, left_cannibals, right_missionaries, right_cannibals) < 0:
        return False

    if left_missionaries > 0 and left_cannibals > left_missionaries:
        return False

    if right_missionaries > 0 and right_cannibals > right_missionaries:
        return False

    return True

### Função sucessor
É listado todos os movimentos possíveis de acordo com a quantidade de lugares no barco, por meio da soma do valor dos index, é possível comparar quantas pessoas de cada cultura cabe no espaço do barco.

In [143]:
def possible_moves():
    moves = []
    
    for missionary_index in range(BOAT_CAPACITY + 1):
        for cannibal_index in range(BOAT_CAPACITY + 1):
            total_in_boat = missionary_index + cannibal_index
            
            if 1 <= total_in_boat <= BOAT_CAPACITY:
                moves.append((missionary_index, cannibal_index))
    
    return moves

In [144]:
possible_moves()

[(0, 1), (0, 2), (0, 3), (1, 0), (1, 1), (1, 2), (2, 0), (2, 1), (3, 0)]

Com isso, podemos finalmente criar a função que aplica os movimentos:

In [145]:
def apply_move(state, move):
    missionaries, cannibals = move
    left_missionaries, left_cannibals = state["left"]
    right_missionaries, right_cannibals = state["right"]

    if state["boat"] == "left":
        return {
            "left": (left_missionaries - missionaries, left_cannibals - cannibals),
            "right": (right_missionaries + missionaries, right_cannibals + cannibals),
            "boat": "right"
        }
    else:
        return {
            "left": (left_missionaries + missionaries, left_cannibals + cannibals),
            "right": (right_missionaries - missionaries, right_cannibals - cannibals),
            "boat": "left"
        }

In [146]:
# Test
new_state = apply_move(initial_state, (1, 1))
print(new_state)
print("Válido?", is_valid(new_state))

{'left': (3, 3), 'right': (1, 1), 'boat': 'right'}
Válido? True


### Explicação da função sucessor
A função sucessor é responsável por gerar, a partir de um estado atual, todos os próximos estados possíveis que podem ser alcançados em **uma única travessia** do barco. Para isso, ela combina três etapas já definidas anteriormente:

1. Obtém todos os movimentos possíveis por meio da função `possible_moves()` (combinações de missionários e canibais que cabem no barco).
2. Aplica cada movimento ao estado atual com a função `apply_move(state, move)`, produzindo um novo estado candidato.
3. Valida esse novo estado com `is_valid(new_state)`, descartando automaticamente estados que violam as regras do problema.

Ao final, a função retorna uma lista de pares `(movimento, novo_estado)`. Essa estrutura é essencial para os algoritmos de busca (como o BFS), pois permite explorar o espaço de estados de forma sistemática, mantendo apenas transições válidas e seguras.

In [147]:
def successors(state):
    result = []

    for move in possible_moves():
        new_state = apply_move(state, move)

        if is_valid(new_state):
            result.append((move, new_state))

    return result

In [148]:
def is_goal(state):
    return state["left"] == (0, 0)

## BFS - Busca em Largura

In [149]:
from collections import deque

def solve_bfs(initial_state):
    queue = deque([{
        "state": initial_state,
        "path": [],
        "movements": [(0,0)]
    }])

    visited = {
        (
            initial_state["left"],
            initial_state["right"],
            initial_state["boat"]
        )
    }

    while queue:
        node = queue.popleft()

        state = node["state"]
        path = node["path"]
        movements = node["movements"]

        if is_goal(state):
            return path + [state], movements

        #obtem uma lista de movimentos e estados que esses movimentos geram
        for move, new_state in successors(state):
            state_key = (
                new_state["left"],
                new_state["right"],
                new_state["boat"]
            )

            # verifica se o estado analisado é um estado que nunca foi visitado, 
            # se não for, esse estado é adicionado na fila pra processamento futuro
            if state_key not in visited:
                visited.add(state_key)

                queue.append({
                    "state": new_state,
                    "path": path + [state],
                    "movements": movements + [move]
                })

    return None, None

In [150]:
solution, movements = solve_bfs(initial_state)

if solution and movements:
    print("Estados da solução (passo a passo):")
    for step, (state, move) in enumerate(zip(solution, movements)):
        print(f"Passo {step}")
        print(state, "através de", move)
        print("-" * 30)
    print("Chegamos")

    print("\nGabarito da travessia:")
    boat_side = "left"

    for step, move in enumerate(movements[1:], start=1):
        missionaries, cannibals = move
        origin = "esquerda" if boat_side == "left" else "direita"
        destination = "direita" if boat_side == "left" else "esquerda"

        missionaries_label = "missionário" if missionaries == 1 else "missionários"
        cannibals_label = "canibal" if cannibals == 1 else "canibais"

        print(
            f"Passo {step}: levar {missionaries} {missionaries_label} e "
            f"{cannibals} {cannibals_label} da margem {origin} para a margem {destination}."
        )

        boat_side = "right" if boat_side == "left" else "left"
else:
    print("Nenhuma solução encontrada")

Estados da solução (passo a passo):
Passo 0
{'left': (4, 4), 'right': (0, 0), 'boat': 'left'} através de (0, 0)
------------------------------
Passo 1
{'left': (4, 2), 'right': (0, 2), 'boat': 'right'} através de (0, 2)
------------------------------
Passo 2
{'left': (4, 3), 'right': (0, 1), 'boat': 'left'} através de (0, 1)
------------------------------
Passo 3
{'left': (4, 0), 'right': (0, 4), 'boat': 'right'} através de (0, 3)
------------------------------
Passo 4
{'left': (4, 1), 'right': (0, 3), 'boat': 'left'} através de (0, 1)
------------------------------
Passo 5
{'left': (1, 1), 'right': (3, 3), 'boat': 'right'} através de (3, 0)
------------------------------
Passo 6
{'left': (2, 2), 'right': (2, 2), 'boat': 'left'} através de (1, 1)
------------------------------
Passo 7
{'left': (0, 2), 'right': (4, 2), 'boat': 'right'} através de (2, 0)
------------------------------
Passo 8
{'left': (0, 3), 'right': (4, 1), 'boat': 'left'} através de (0, 1)
----------------------------

## DFS - Busca em Profundidade
A busca em profundidade explora um caminho até o fim antes de voltar (backtracking) para tentar alternativas. Aqui, usamos uma pilha para controlar os estados pendentes e também evitamos revisitar estados já explorados.

In [151]:
def solve_dfs(initial_state):
    stack = [{
        "state": initial_state,
        "path": [],
        "movements": [(0, 0)]
    }]

    visited = {
        (
            initial_state["left"],
            initial_state["right"],
            initial_state["boat"]
        )
    }

    while stack:
        node = stack.pop()

        state = node["state"]
        path = node["path"]
        movements = node["movements"]

        if is_goal(state):
            return path + [state], movements

        for move, new_state in successors(state):
            state_key = (
                new_state["left"],
                new_state["right"],
                new_state["boat"]
            )

            if state_key not in visited:
                visited.add(state_key)
                stack.append({
                    "state": new_state,
                    "path": path + [state],
                    "movements": movements + [move]
                })

    return None, None

In [152]:
solution_dfs, movements_dfs = solve_dfs(initial_state)

if solution_dfs and movements_dfs:
    print("Estados da solução em DFS (passo a passo):")
    for step, (state, move) in enumerate(zip(solution_dfs, movements_dfs)):
        print(f"Passo {step}")
        print(state, "através de", move)
        print("-" * 30)
    print("Chegamos")

    print("\nGabarito da travessia (DFS):")
    boat_side = "left"

    for step, move in enumerate(movements_dfs[1:], start=1):
        missionaries, cannibals = move
        origin = "esquerda" if boat_side == "left" else "direita"
        destination = "direita" if boat_side == "left" else "esquerda"

        missionaries_label = "missionário" if missionaries == 1 else "missionários"
        cannibals_label = "canibal" if cannibals == 1 else "canibais"

        print(
            f"Passo {step}: levar {missionaries} {missionaries_label} e "
            f"{cannibals} {cannibals_label} da margem {origin} para a margem {destination}."
        )

        boat_side = "right" if boat_side == "left" else "left"
else:
    print("Nenhuma solução encontrada com DFS")

Estados da solução em DFS (passo a passo):
Passo 0
{'left': (4, 4), 'right': (0, 0), 'boat': 'left'} através de (0, 0)
------------------------------
Passo 1
{'left': (3, 3), 'right': (1, 1), 'boat': 'right'} através de (1, 1)
------------------------------
Passo 2
{'left': (4, 3), 'right': (0, 1), 'boat': 'left'} através de (1, 0)
------------------------------
Passo 3
{'left': (2, 2), 'right': (2, 2), 'boat': 'right'} através de (2, 1)
------------------------------
Passo 4
{'left': (3, 3), 'right': (1, 1), 'boat': 'left'} através de (1, 1)
------------------------------
Passo 5
{'left': (0, 3), 'right': (4, 1), 'boat': 'right'} através de (3, 0)
------------------------------
Passo 6
{'left': (0, 4), 'right': (4, 0), 'boat': 'left'} através de (0, 1)
------------------------------
Passo 7
{'left': (0, 1), 'right': (4, 3), 'boat': 'right'} através de (0, 3)
------------------------------
Passo 8
{'left': (1, 1), 'right': (3, 3), 'boat': 'left'} através de (1, 0)
---------------------

## DLS - Busca em Profundidade Limitada
A busca em profundidade limitada (Depth-Limited Search) funciona como a DFS, mas com um limite máximo de profundidade. Isso evita que a busca avance indefinidamente em ramos muito longos e permite controlar o custo de exploração.

In [153]:
def solve_dls(initial_state, depth_limit):
    def state_to_key(state):
        return (state["left"], state["right"], state["boat"])

    initial_key = state_to_key(initial_state)
    stack = [{
        "state": initial_state,
        "path": [],
        "movements": [(0, 0)],
        "depth": 0,
        "path_keys": {initial_key}
    }]

    while stack:
        node = stack.pop()

        state = node["state"]
        path = node["path"]
        movements = node["movements"]
        depth = node["depth"]
        path_keys = node["path_keys"]

        if is_goal(state):
            return path + [state], movements

        if depth >= depth_limit:
            continue

        for move, new_state in successors(state):
            state_key = state_to_key(new_state)

            # Evita ciclos no caminho atual da busca em profundidade
            if state_key in path_keys:
                continue

            stack.append({
                "state": new_state,
                "path": path + [state],
                "movements": movements + [move],
                "depth": depth + 1,
                "path_keys": path_keys | {state_key}
            })

    return None, None

In [154]:
depth_limit = 0
max_depth_limit = 500
solution_dls, movements_dls = None, None

while depth_limit <= max_depth_limit:
    solution_dls, movements_dls = solve_dls(initial_state, depth_limit)

    if solution_dls is None:
        print(f"depth_limit = {depth_limit}: nenhuma solução encontrada.")
        depth_limit += 1
        continue

    print(f"depth_limit = {depth_limit}: solução encontrada!")
    break

if solution_dls is None:
    print(
        f"\nNenhuma solução encontrada até depth_limit = {max_depth_limit}."
    )
else:
    print(f"\nSolução final encontrada com depth_limit = {depth_limit}")
    print(f"Estados da solução em DLS (limite = {depth_limit}):")

    for step, (state, move) in enumerate(zip(solution_dls, movements_dls)):
        print(f"Passo {step}")
        print(state, "através de", move)
        print("-" * 30)
    print("Chegamos")

    print("\nGabarito da travessia (DLS):")
    boat_side = "left"

    for step, move in enumerate(movements_dls[1:], start=1):
        missionaries, cannibals = move
        origin = "esquerda" if boat_side == "left" else "direita"
        destination = "direita" if boat_side == "left" else "esquerda"

        missionaries_label = "missionário" if missionaries == 1 else "missionários"
        cannibals_label = "canibal" if cannibals == 1 else "canibais"

        print(
            f"Passo {step}: levar {missionaries} {missionaries_label} e "
            f"{cannibals} {cannibals_label} da margem {origin} para a margem {destination}."
        )

        boat_side = "right" if boat_side == "left" else "left"

depth_limit = 0: nenhuma solução encontrada.
depth_limit = 1: nenhuma solução encontrada.
depth_limit = 2: nenhuma solução encontrada.
depth_limit = 3: nenhuma solução encontrada.
depth_limit = 4: nenhuma solução encontrada.
depth_limit = 5: nenhuma solução encontrada.
depth_limit = 6: nenhuma solução encontrada.
depth_limit = 7: nenhuma solução encontrada.
depth_limit = 8: nenhuma solução encontrada.
depth_limit = 9: solução encontrada!

Solução final encontrada com depth_limit = 9
Estados da solução em DLS (limite = 9):
Passo 0
{'left': (4, 4), 'right': (0, 0), 'boat': 'left'} através de (0, 0)
------------------------------
Passo 1
{'left': (3, 3), 'right': (1, 1), 'boat': 'right'} através de (1, 1)
------------------------------
Passo 2
{'left': (4, 3), 'right': (0, 1), 'boat': 'left'} através de (1, 0)
------------------------------
Passo 3
{'left': (2, 2), 'right': (2, 2), 'boat': 'right'} através de (2, 1)
------------------------------
Passo 4
{'left': (3, 3), 'right': (1, 1), 

## Parametrização TODO
Você pode alterar o problema sem editar várias células manualmente usando a função `configure_problem` da célula abaixo.

Exemplo de uso:
- `configure_problem(total_missionaries=4, total_cannibals=4, boat_capacity=2)`
- Depois, execute novamente as células dos algoritmos (BFS, DFS e DLS).